# 01 · CFTR2 — the disease-specific *functional* truth set

[CFTR2](https://cftr2.org) classifies *CFTR* variants using **patient outcomes + in-vitro CFTR function assays** — a different, more *functional* kind of evidence than ClinVar's clinical assertions. That functional axis makes it a **partially orthogonal** truth set (section 2 below spells out why "partially" is the honest word). This notebook builds the full CFTR2 release locally from a manually-downloaded workbook (gitignored) — see the build cell below.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · CFTR2 — a disease-specific, *functional* reference

[CFTR2](https://cftr2.org) is different in kind from ClinVar. It is a **CF-specific** database that classifies *CFTR* variants as:

- **CF-causing**
- **Varying clinical consequence** (formerly "CF-causing (mild)")
- **Non CF-causing**
- **No interpretation available** — not yet enough evidence

Crucially, CFTR2's calls are built from **two kinds of evidence together**:
1. **Patient data** — real clinical outcomes across thousands of people with CF who carry the variant.
2. **In-vitro CFTR function assays** — measuring, in the lab, how much working chloride-channel the variant protein actually produces.

That second, functional axis makes CFTR2 **partially orthogonal** to ClinVar — but *not* independent of it (the two share clinical evidence and cross-cite; see section 2).

### Building the data — a manual download (no API)

CFTR2 has no API — the variant list is published as an Excel workbook. **You must
fetch this one yourself:**

1. Go to <https://cftr2.org>, find the variant-list history / download page, and
   get the current release.
2. Save it into `data/` (gitignored — never commit it) and set `CFTR2_XLSX_NAME`
   below to match its filename.

**Version control.** CFTR2 has no historical archive to pin against the way
ClinVar does (checked directly against cftr2.org: there is no dated-release
listing, just the current file). What the cell below does instead:

- Reads the release date straight out of the workbook's own header metadata
  (the `Date:` row CFTR2 ships in every release) rather than assuming one.
- Persists that date into `data/cftr2_cftr.release.json` alongside the extract,
  so `load_cftr2()` can expose it as a `cftr2_release` column.
- If you want to reproduce a *past* run, you need to have manually saved that
  older workbook yourself (cftr2.org doesn't keep one for you) — point
  `CFTR2_XLSX_NAME` at it, and the recorded release date will reflect whatever
  that file's own header says, not today's date.

The cell reads two sheets from the workbook: **"CFTR2 variants by legacy
name"** (the variant list itself — legacy name, protein name, cDNA name, allele
count/frequency, and functional class) and **"Genomic coordinates"** (authoritative
GRCh38 positions). It derives the 1-letter `protein_variant` key for simple
single-residue missense variants only (regex on the protein name), resolves a
handful of variants listed under a **pipe-combined cDNA name** (e.g. W1282X is
`c.3845G>A|c.3846G>A`, two SNVs that create the same stop codon) by trying each
`|`-separated alternative against the genomic sheet, and writes
`data/cftr2_cftr.csv`. It also asserts the workbook's header states the
expected MANE transcript (`NM_000492.4`) before trusting its coordinates.

License: CFTR2's public data-use terms (cite CFTR2 if you use it) — see
`data_manifest.json`.

In [2]:
import re, json, openpyxl
from datetime import datetime, timezone

DATA_DIR = pathlib.Path.cwd().parent / "data"
# Change this if you've manually sourced a different (e.g. older) CFTR2 release --
# whatever release date IS in that file's own header is what gets recorded.
CFTR2_XLSX_NAME = "CFTR2_30January2026.xlsx"
CFTR2_XLSX = DATA_DIR / CFTR2_XLSX_NAME
CFTR2_TSV = DATA_DIR / "cftr2_cftr.csv"
CFTR2_RELEASE_JSON = DATA_DIR / "cftr2_cftr.release.json"
EXPECT_TX = "NM_000492.4"
AA3TO1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q",
    "Glu": "E", "Gly": "G", "His": "H", "Ile": "I", "Leu": "L", "Lys": "K",
    "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S", "Thr": "T", "Trp": "W",
    "Tyr": "Y", "Val": "V",
}
MIS = re.compile(r"^p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})$")   # simple single-residue missense only


def missense_key(protein_name: str) -> str:
    if not protein_name:
        return ""
    m = MIS.match(protein_name.strip())
    if not m:
        return ""
    a, pos, b = m.group(1), m.group(2), m.group(3)
    return f"{AA3TO1[a]}{pos}{AA3TO1[b]}" if a in AA3TO1 and b in AA3TO1 else ""


if CFTR2_TSV.exists():
    print(f"already built -> {CFTR2_TSV.name} (delete it and {CFTR2_RELEASE_JSON.name} to rebuild)")
elif not CFTR2_XLSX.exists():
    raise FileNotFoundError(
        f"{CFTR2_XLSX} not found.\n"
        "CFTR2 has no API -- get the variant-list workbook:\n"
        "  1. Go to https://cftr2.org and download the current release xlsx\n"
        f"  2. Save it as {CFTR2_XLSX} (do NOT commit it -- data/ is gitignored)\n"
        "     (or set CFTR2_XLSX_NAME above to whatever you saved it as)\n"
        "Then re-run this cell."
    )
else:
    wb = openpyxl.load_workbook(CFTR2_XLSX, read_only=True, data_only=True)
    ws = wb["CFTR2 variants by legacy name"]

    # Header rows (1-12) carry provenance: release date, official counts, and --
    # critically -- the reference transcript, so the build documents its own basis
    # instead of assuming GRCh38/MANE.
    header_meta = {}
    for r in ws.iter_rows(min_row=1, max_row=12, values_only=True):
        cell = str(r[0]).strip() if r[0] is not None else ""
        if ":" in cell:
            k, v = cell.split(":", 1)
            header_meta[k.strip()] = v.strip()
    tx = header_meta.get("CFTR reference transcript", "")
    assert EXPECT_TX in tx, (
        f"CFTR2 header transcript is {tx!r}, expected {EXPECT_TX}; the extract's "
        "genomic coordinates + MANE assumptions may no longer hold -- check the release.")
    print("CFTR2 header provenance:")
    for k in ("Date", "Number of patients in CFTR2", "Number of variants reported in CFTR2",
              "Number of variants with interpretations", "CFTR reference transcript"):
        if k in header_meta:
            print(f"  {k}: {header_meta[k]}")

    rows = []
    for r in ws.iter_rows(min_row=13, values_only=True):
        if r[0] is None:
            continue
        legacy, protein, cdna, alt, alleles, af, prev, cur, changed = r[:9]
        rows.append({"protein_variant": missense_key(protein or ""), "legacy_name": legacy,
                     "protein_name": protein, "cdna_name": cdna, "cftr2_alleles": alleles,
                     "cftr2_af": af, "cftr2_class": cur})
    df = pd.DataFrame(rows)

    # Merge GRCh38 genomic coordinates from sheet 2 (on cDNA name).
    ws2 = wb["Genomic coordinates"]
    g = pd.DataFrame(ws2.iter_rows(min_row=2, values_only=True),
                      columns=list(next(ws2.iter_rows(min_row=1, max_row=1, values_only=True))))
    gcols = {"Variant cDNA name": "cdna_name", "grch38_chr": "grch38_chr",
             "grch38_pos": "grch38_pos", "grch38_ref": "grch38_ref", "grch38_alt": "grch38_alt"}
    g = g[list(gcols)].rename(columns=gcols).drop_duplicates("cdna_name")
    gcoord = {k: v for k, v in g.set_index("cdna_name")
              [["grch38_chr", "grch38_pos", "grch38_ref", "grch38_alt"]].to_dict("index").items()}

    def resolve_coords(cdna):
        # Some CFTR2 variants are listed under a PIPE-combined cDNA name (e.g. W1282X is
        # 'c.3845G>A|c.3846G>A') but the genomic sheet keys each single name separately.
        # Try the whole name, then each alternative, taking the first with coordinates.
        if not isinstance(cdna, str):
            return {}
        for alt in [cdna, *cdna.split("|")]:
            hit = gcoord.get(alt.strip())
            if hit and pd.notna(hit.get("grch38_pos")):
                return hit
        return {}

    coords = pd.DataFrame([resolve_coords(c) for c in df["cdna_name"]], index=df.index,
                           columns=["grch38_chr", "grch38_pos", "grch38_ref", "grch38_alt"])
    df = pd.concat([df, coords], axis=1)
    df.to_csv(CFTR2_TSV, index=False)

    CFTR2_RELEASE_JSON.write_text(json.dumps({
        "release_date": header_meta.get("Date", "unknown"),
        "source_xlsx": CFTR2_XLSX_NAME,
        "built_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "row_count": len(df),
    }, indent=2))

    print(f"\nwrote {CFTR2_TSV.relative_to(DATA_DIR.parent)} and {CFTR2_RELEASE_JSON.name}  rows: {len(df):,}")
    print("with a 1-letter missense key:", (df["protein_variant"] != "").sum(),
          f"(of {len(df)}; the rest are non-missense -- join by cdna_name/genomic coords)")

already built -> cftr2_cftr.csv (delete it and cftr2_cftr.release.json to rebuild)


In [3]:
cftr2 = tk.load_cftr2()          # the built extract, once you've run the build cell above
print('source :', cftr2['source'].unique(), '| release:', cftr2['cftr2_release'].iloc[0])
print('variants:', len(cftr2), '| with a missense key:', (cftr2['protein_variant'] != '').sum())
print()
print(cftr2['cftr2_class'].value_counts().to_string())
cftr2.head(8)

source : ['REAL'] | release: 30 January 2026
variants: 2097 | with a missense key: 780

cftr2_class
CF-causing                      1245
No interpretation available      722
Varying clinical consequence      83
Non CF-causing                    42


,protein_variant,legacy_name,protein_name,cdna_name,cftr2_alleles,cftr2_af,cftr2_class,grch38_chr,grch38_pos,grch38_ref,grch38_alt,cftr2_release,source
0,,F508del,p.Phe508del,c.1521_1523del,137363,0.650682595473364,CF-causing,7.0,117559590.0,ATCT,A,30 January 2026,REAL
1,,G542X,p.Gly542X,c.1624G>T,5752,0.027246975453089916,CF-causing,7.0,117587778.0,G,T,30 January 2026,REAL
2,G551D,G551D,p.Gly551Asp,c.1652G>A,3831,0.01814728146049852,CF-causing,7.0,117587806.0,G,A,30 January 2026,REAL
3,N1303K,N1303K,p.Asn1303Lys,c.3909C>G,3551,0.01682093355944407,CF-causing,7.0,117652877.0,C,G,30 January 2026,REAL
4,,W1282X,p.Trp1282X,c.3845G>A|c.3846G>A,2500,0.011842391973700416,CF-causing,7.0,117642565.0,G,A,30 January 2026,REAL
5,R117H,R117H,p.Arg117His,c.350G>A,2262,0.010714996257804137,Varying clinical consequence,7.0,117530975.0,G,A,30 January 2026,REAL
6,,3849+10kbC->T,p.?,c.3718-2477C>T,1990,0.00942654401106553,CF-causing,7.0,117639961.0,C,T,30 January 2026,REAL
7,,621+1G->T,p.?,c.489+1G>T,1860,0.008810739628433109,CF-causing,7.0,117531115.0,G,T,30 January 2026,REAL


### Which variants get a `protein_variant` key — and what are the rest?

The 1-letter `protein_variant` key (e.g. `G551D`) is derived by the build cell above from
the protein name with a regex for *simple single-residue missense* only. It exists for
~780 of ~2,097 variants. **The other ~1,317 are NOT all splice variants** — most are
deletions and nonsense. They carry an empty key and must be joined by `cdna_name` or
genomic coordinates instead.

In [4]:
import re
cf = cftr2.copy()
cf["has_key"] = cf["protein_variant"].fillna("") != ""
print("with missense key:", int(cf["has_key"].sum()), "| without:", int((~cf["has_key"]).sum()))

def category(row):
    p, c = str(row.get("protein_name") or ""), str(row.get("cdna_name") or "")
    if "del" in p or "del" in c: return "deletion/indel"
    if "X" in p or "Ter" in p:   return "nonsense (stop-gain)"
    if "ins" in c or "dup" in c: return "insertion/dup"
    if ("+" in c) or ("-" in c and "c." in c): return "splice/intronic"
    if "=" in p: return "synonymous"
    return "other/complex"

nokey = cf[~cf["has_key"]]
print("\nWhat the NON-missense (no-key) variants actually are:")
print(nokey.apply(category, axis=1).value_counts().to_string())

with missense key: 780 | without: 1317

What the NON-missense (no-key) variants actually are:
deletion/indel          540
nonsense (stop-gain)    347
splice/intronic         291
other/complex            50
insertion/dup            48
synonymous               41


## 2 · How orthogonal is CFTR2, really?

CFTR2's **functional-assay** component (in-vitro chloride-channel measurements)
is a wet-lab signal no sequence model trained on — genuinely independent
evidence. But CFTR2 is **not** an independent gold standard: its
**patient/clinical** component overlaps the same evidence that feeds ClinVar,
**ClinVar entries cite CFTR2**, and **CFTR2 informs the ACMG CFTR guidance**
ClinVar submitters follow — the two databases cross-reference each other.

> **Rule of thumb:** use ClinVar for **breadth**; lean on CFTR2's *functional*
> measurements as **partial** orthogonal evidence — but never report "agrees
> with CFTR2" as if it were independent of ClinVar.

The same caution has a second half worth stating: even a genuinely orthogonal
truth set doesn't rescue a benchmark from **temporal leakage**. A variant
described in the literature decades before a model was trained is in that
model's training data; scoring it correctly demonstrates recall, not skill. A
defensible benchmark therefore needs *both* an orthogonal truth set *and* a
training-cutoff hold-out — see [`../tools/05_revel.ipynb`](../tools/05_revel.ipynb) §2.

## 3 · When did CFTR2 first call each variant CF-causing?

Section 2 ends by saying a defensible benchmark needs a **training-cutoff hold-out**.
That needs a per-variant date, and this section builds one.

Without it, "the tool got this right" and "the tool memorised it" are the same
observation. A variant whose disease-causing status was public years before a model was
trained is in that model's training data; scoring it correctly demonstrates recall. Only
variants whose status became known *after* a model's cutoff can distinguish the two.

### What "the date" means here

CFTR2's workbook has **no per-variant date field** — only the `Date:` row in its header.
So the date used here is a derived one, and it is worth being exact about what it measures:

> **the first CFTR2 release whose determination for this variant reads `CF-causing`.**

That is measurable because every CFTR2 release carries **two** determination columns,
current and previous, and each release's previous-version column names the release before
it. Chained across the workbooks in `data/`, they give a per-variant trajectory.

This is CFTR2's *own* assertion — nothing is borrowed from ClinVar or the literature, and
nothing is invented. What it costs is stated below.

### The floor, and what it does to F508del

The chain reaches back only as far as the oldest workbook's previous-version column. Every
variant already CF-causing at that point gets `left_censored`: the value is a **bound, not
a date**. F508del is one of them. F508del was reported in 1989 — the date this section can
prove is roughly a quarter-century later, and no amount of chaining recovers the
difference. Treat `left_censored` as "known before the archive begins", never as a date.

That is still enough to be useful: every tool in `toolkit.TOOL_YEAR` was released after the
floor, so a hold-out is constructible for all of them and the censored bucket is a clean
category — "known before every tool in this repo".

### Reproducibility

cftr2.org publishes no dated-release archive, so this section reads workbooks that were
saved locally as each release came out. **A fresh clone cannot rebuild it**, and unlike the
single-release build above there is no way to obtain the missing files after the fact.

In [5]:
import cftr2_history as ch

CFTR2_HIST_CSV = DATA_DIR / "cftr2_history.csv"
# Every CFTR2 workbook saved locally. The module orders them by each file's own `Date:`
# header, so the filenames carry no meaning and a new release just needs dropping in.
WORKBOOKS = sorted(DATA_DIR.glob("CFTR2_*.xlsx"))

if CFTR2_HIST_CSV.exists():
    print(f"already built -> {CFTR2_HIST_CSV.name} "
          f"(delete it and cftr2_history.release.json to rebuild)")
elif len(WORKBOOKS) < 2:
    raise FileNotFoundError(
        f"found {len(WORKBOOKS)} CFTR2 workbook(s) in {DATA_DIR}; the release series needs "
        "several. cftr2.org publishes only the current file, so these cannot be fetched "
        "retrospectively -- this section is unbuildable without locally archived releases.")
else:
    hist = ch.build_history(WORKBOOKS)
    meta = hist["meta"]
    print(f"releases chained: {meta['release_count']}  "
          f"({meta['releases'][0]} -> {meta['latest_release']})")
    print(f"censoring floor : {meta['censoring_floor']}  "
          f"(from the oldest workbook's previous-version column)")
    print(f"join key        : {meta['join_key']}   renames merged: {meta['renames_applied']}")
    print(f"\nchain verified contiguous across {len(meta['chain'])} steps -- "
          "each release's previous-version label matches the preceding file's own Date:")
    for step in meta["chain"]:
        print(f"  {step}")
    paths = ch.write_extracts(hist, DATA_DIR)
    print(f"\nwrote {paths['summary'].name} ({len(hist['summary']):,} variants), "
          f"{paths['long'].name} ({len(hist['long']):,} variant-releases) "
          f"and {paths['release'].name}")

releases chained: 12  (2016-08-08 -> 2026-01-30)
censoring floor : 2015-08-13  (from the oldest workbook's previous-version column)
join key        : legacy_name   renames merged: 7

chain verified contiguous across 11 steps -- each release's previous-version label matches the preceding file's own Date:
  2016-08-08 -> 2017-03-17
  2017-03-17 -> 2017-12-08
  2017-12-08 -> 2018-08-31
  2018-08-31 -> 2019-03-11
  2019-03-11 -> 2020-01-10
  2020-01-10 -> 2020-07-31
  2020-07-31 -> 2021-09-24
  2021-09-24 -> 2022-04-29
  2022-04-29 -> 2023-04-07
  2023-04-07 -> 2024-09-25
  2024-09-25 -> 2026-01-30

wrote cftr2_history.csv (2,102 variants), cftr2_history_long.csv (7,978 variant-releases) and cftr2_history.release.json


### Why the join key is the legacy name

Chaining twelve releases is a join, and the join is where this goes wrong quietly. Three
things in the source would each return a plausible wrong answer rather than an error, so
`cftr2_history.py` guards all three and **raises**:

| what changes | what it would do undetected | guard |
|---|---|---|
| **2023 nomenclature migration** — `c.1029delC` → `c.1029del`, `c.1021_1022dupTC` → `c.1021_1022dup`, complex alleles bracketed | keying on cDNA name loses variants at that step and re-dates long-known alleles to 2023 | dropout rate per release step |
| **class vocabulary drift** — `Unknown significance` → `No interpretation available` | a pure relabelling reads as a reclassification for every affected variant | both map to one normalised class; the raw string is kept |
| **rename tombstones** — `(CF-causing under new name)` | scored as a class it invents downgrades; dropped outright it orphans the successor, which then looks new | rename target parsed, old name aliased to new, trajectories merged |

The **legacy name** is the only identifier stable across all twelve releases, and it is
unique within every one of them — so it is the key. The cDNA name is not: it is the thing
the 2023 migration rewrote.

The dropout guard is the one that matters, and the more obvious "did a variant vanish and
come back?" check is **not** a substitute for it. When a key breaks, most affected variants
do not vanish and return — they vanish under the old name and reappear under a new one, so
they read as one cohort dropped and another added, and every one is silently re-dated. Only
the few that happen to revert would trip a resurrection check.

The cell below runs the build again deliberately keyed on the cDNA name, to show the guard
firing on the failure it exists to catch rather than merely asserting that it would.

In [6]:
# NEGATIVE CONTROL -- this build is SUPPOSED to fail.
# Guard messages can name variants; CFTR2 names are not republished here, so anything
# after an "(e.g. ..." is cut before printing.
def redact(err):
    return str(err).split("(e.g.")[0].strip()

try:
    ch.build_history(WORKBOOKS, key_by="cdna_name")
    print("NO GUARD FIRED -- the negative control did not fail, which means the guard "
          "is no longer protecting anything. Do not trust the dates.")
except ch.HistoryError as e:
    print("guard fired as intended, keyed on cDNA name:\n")
    print(" ", redact(e))

# ...and the same build on the real key must succeed, or the comparison proves nothing.
ch.build_history(WORKBOOKS)
print("\nsame twelve workbooks keyed on legacy name: builds cleanly.")

guard fired as intended, keyed on cDNA name:

  39 of 400 variants present at 2018-08-31 are absent at 2019-03-11 (limit 5). CFTR2 does not retire variants in bulk, so the join key has almost certainly broken: those variants are still in the list under a changed name and every one of them would be re-dated to 2019-03-11. Check the key column against that release's nomenclature.



same twelve workbooks keyed on legacy name: builds cleanly.


### Attaching the dates to the benchmark set

`load_cftr2_history()` returns one row per variant with the derived date and, separately,
**how to read it** — `cftr2_date_basis` is a column, not something folded into the date:

| `cftr2_date_basis` | meaning |
|---|---|
| `observed` | a real date from the release series |
| `left_censored` | already CF-causing at the floor — a **bound**, not a date |
| `never_cf_causing` | CFTR2 has never called it CF-causing |

The per-release trajectory behind the summary is kept too, in `cftr2_history_long.csv`,
one row per variant per release — so a reclassification can be inspected rather than
inferred from a collapsed answer.

Two joining details worth knowing, both checked by the cell below. The join is on the
legacy name for the reason given above; and CFTR2's workbook carries one legacy name with
a trailing non-breaking space, so both sides are whitespace-normalised. The remaining
unmatched rows are the workbook's trailing footnote and copyright lines, which sit in the
same column as variant names — CFTR2's own header states **2,092** variants.

In [7]:
history = tk.load_cftr2_history()

# Join on the legacy name (the only cross-release-stable key), whitespace-normalised on
# both sides. cdna_name is dropped from the right so the extract's own copy is kept.
key = cftr2["legacy_name"].astype("string").str.strip()
dated = (cftr2.assign(_key=key)
               .merge(history.drop(columns=["cdna_name"]),
                      left_on="_key", right_on="variant_key", how="left"))
assert len(dated) == len(cftr2), "join fanned out -- the key is not unique on one side"

undated = dated["cftr2_date_basis"].isna()
real = dated[~undated]
print(f"benchmark rows {len(dated):,} | dated {len(real):,} | "
      f"unmatched {int(undated.sum())} (the workbook's footnote/copyright lines)")

# Cross-check against CFTR2's own header tallies -- if the dating disagrees with the
# release it was built from, the dates are wrong, not merely incomplete.
ever_cf = real["cftr2_date_basis"].isin(["observed", "left_censored"]).sum()
assert int(ever_cf) == int((real["cftr2_class"] == "CF-causing").sum()) == 1245, "header cross-check failed"
print(f"cross-check: {ever_cf:,} variants ever CF-causing == the 2026 workbook header's 1,245\n")

print(real["cftr2_date_basis"].value_counts().to_string())

benchmark rows 2,097 | dated 2,092 | unmatched 5 (the workbook's footnote/copyright lines)
cross-check: 1,245 variants ever CF-causing == the 2026 workbook header's 1,245

cftr2_date_basis
observed            1007
never_cf_causing     847
left_censored        238


In [8]:
first_cf = pd.to_datetime(real.loc[real["cftr2_date_basis"] == "observed",
                                   "cftr2_first_cf_causing"])

print("first called CF-causing, by release:")
print(first_cf.dt.date.value_counts().sort_index().to_string())

print("\nhow the determination moved over the series:")
print(f"  reclassified at least once   : {int((real['cftr2_class_changes'] > 0).sum())}")
print(f"  ever withdrawn from CF-causing: {int(real['cftr2_ever_withdrawn'].sum())}"
      "   (a round trip counts; pair with cftr2_current_determination)")
print(f"  never CF-causing              : {int((real['cftr2_date_basis'] == 'never_cf_causing').sum())}"
      "   -- the candidate negatives")

first called CF-causing, by release:
cftr2_first_cf_causing
2016-08-08     30
2017-03-17      9
2017-12-08     32
2018-08-31     24
2019-03-11     11
2020-01-10      7
2020-07-31      8
2021-09-24     21
2022-04-29     18
2023-04-07    316
2024-09-25    367
2026-01-30    164

how the determination moved over the series:
  reclassified at least once   : 23
  ever withdrawn from CF-causing: 1   (a round trip counts; pair with cftr2_current_determination)
  never CF-causing              : 847   -- the candidate negatives


### What this buys: a hold-out per tool

For each predictor, the variants CFTR2 first called CF-causing **after** that tool was
released could not have been in its training data as CF-causing. Those are the ones where a
correct score is evidence of skill rather than recall.

`TOOL_YEAR` records release years, so the comparison below is by year and deliberately
conservative: a variant is counted only if its first CF-causing release falls in a year
*strictly after* the tool's. Same-year cases are excluded rather than guessed at.

Note which row matters most. `LABEL_SUPERVISED` marks the tools trained directly on curated
clinical labels — leakage is a first-order problem for those and a second-order one for the
rest, which never saw clinical labels at all and can only leak through the literature that
informed them.

In [9]:
holdout = pd.DataFrame(
    [{"tool": t,
      "released": yr,
      "label_supervised": tk.LABEL_SUPERVISED[t],
      "cf_causing_after_release": int((first_cf.dt.year > yr).sum()),
      "leaked_or_censored": int(len(real[real["cftr2_date_basis"].isin(["observed", "left_censored"])])
                                - (first_cf.dt.year > yr).sum())}
     for t, yr in sorted(tk.TOOL_YEAR.items(), key=lambda kv: kv[1])])
print(holdout.to_string(index=False))
print(f"\ncandidate negatives available to every tool: "
      f"{int((real['cftr2_date_basis'] == 'never_cf_causing').sum())} never-CF-causing variants")

         tool  released  label_supervised  cf_causing_after_release  leaked_or_censored
        REVEL      2016              True                       977                 268
    PrimateAI      2018             False                       912                 333
     SpliceAI      2019             False                       901                 344
          EVE      2021             False                       865                 380
     Pangolin      2022             False                       847                 398
AlphaMissense      2023             False                       531                 714
        ESM1b      2023             False                       531                 714
         CADD      2024             False                       164                1081

candidate negatives available to every tool: 847 never-CF-causing variants


### What this date can and cannot support

**It can** separate recall from skill for any tool released after the floor, and it says
which variants to use: score only the ones CFTR2 first called CF-causing after the tool
shipped, against the never-CF-causing set as negatives.

**It cannot** tell you when a variant was *discovered*, or when anyone first called it
disease-causing. It measures one thing — when **CFTR2** adopted the call. Three gaps follow
from that, and none of them are closed by more careful chaining:

- **Everything before the floor is one bucket.** `left_censored` variants are ordered
  relative to nothing. F508del and a variant first described in 2014 are indistinguishable.
- **Resolution is the release cadence**, 6–18 months. A variant dated to a release was
  called CF-causing at some point since the previous one.
- **CFTR2 is a follower, not a first mover.** A variant is typically called CF-causing in
  the literature and in ClinVar before CFTR2 adopts it, so an `observed` date is an
  **upper bound** on when the label became public. Used for a hold-out it errs the safe
  way — it can wrongly exclude a variant as leaked, but it will not wrongly admit one.

That last point is the honest limit on the whole section. This dates CFTR2's adoption of a
call, which is a defensible and conservative proxy for when the label was learnable — not
the date the knowledge first existed.

## Key takeaways

1. **CFTR2** calls combine **patient data + functional assays**. The functional axis is *partially* orthogonal to ClinVar — but CFTR2 cross-cites ClinVar, so it is **not** an independent gold standard (see [`../tools/05_revel.ipynb`](../tools/05_revel.ipynb) §2).
2. CFTR2's own header counts **2,092** variants (the built extract has 2,097 rows because five trailing footnote and copyright lines sit in the variant-name column). **780** have a 1-letter missense key, 779 of which join to AlphaMissense; the rest are non-missense — mostly deletions and nonsense, not splice — and join by `cdna_name` / genomic coordinates.
3. **Version:** the build cell reads the release date out of the workbook's own header and persists it as `cftr2_release` — cftr2.org publishes no dated-release archive, so reproducing a past run means having manually saved that older workbook yourself.
4. **Every CF-causing call is dated** (§3) from CFTR2's own release series: `cftr2_first_cf_causing` plus `cftr2_date_basis` saying whether that is a real date, a censoring bound, or absent. This is what makes a training-cutoff hold-out possible — but it dates *CFTR2's adoption* of a call, which is an upper bound on when the label became public, and everything already CF-causing at the 2015-08-13 floor is one undifferentiated bucket.
5. The **CFTR2-vs-ClinVar agreement** cross-check lived in the archived integration notebook.

**The splice predictors carry on from here.** [`../tools/07_spliceai.ipynb`](../tools/07_spliceai.ipynb)
and [`../tools/08_pangolin.ipynb`](../tools/08_pangolin.ipynb) score the non-coding and
synonymous CFTR alleles no missense predictor can reach, and both join onto the CFTR2
coordinates built here. The live CADD API notebook and the cross-tool benchmark over the
whole CFTR2 list are written but held back pending the same audit pass.